# Studio di MERGE
basato su cleaning 4
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
from MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [2]:
file_codes = ['UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS']

#'ADNIMERGE', 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES', 
#            'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS'

### Mixed info ###
# 'ADNIMERGE', 
# 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

### Single Cofactor ###
# 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

### Volumes ###
# 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS',
# 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'  --> just partial immages segmentation

### CSF ###
# 'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [3]:
search = client.query_files(
    query={'custom.level' : 'cleaned_04', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


In [4]:
len(zip_files)

7

In [5]:
file_name_0 = list(zip_files.keys())[0]
df_0 = zip_files[file_name_0].copy(deep=True)

file_name_1 = list(zip_files.keys())[1]
df_1 = zip_files[file_name_1].copy(deep=True)


file_name_2 = list(zip_files.keys())[2]
df_2 = zip_files[file_name_2].copy(deep=True)

file_name_3 = list(zip_files.keys())[3]
df_3 = zip_files[file_name_3].copy(deep=True)

file_name_4 = list(zip_files.keys())[4]
df_4 = zip_files[file_name_4].copy(deep=True)

file_name_5 = list(zip_files.keys())[5]
df_5 = zip_files[file_name_5].copy(deep=True)

file_name_6 = list(zip_files.keys())[6]
df_6 = zip_files[file_name_6].copy(deep=True)

'''
file_name_7 = list(zip_files.keys())[7]
df_7 = zip_files[file_name_7].copy(deep=True)
'''
for x in range(len(zip_files)):
    print(x, '--->', list(zip_files.keys())[x])


0 ---> UCSFFSX_11_02_15_11Aug2025_04.csv
1 ---> UCSFFSX7_11Aug2025_04.csv
2 ---> UCSFFSX6_11Aug2025_04.csv
3 ---> UCSFFSX51_11_08_19_11Aug2025_04.csv
4 ---> UCSFFSL_02_01_16_11Aug2025_04.csv
5 ---> UPENNROI_MARS_06_01_16_09Oct2025_04.csv
6 ---> UCSDVOL_28Oct2025_04.csv


# Confronto stessi RID  ==> RID - EXAMDATE identici tra file

In [6]:
# Lista dei dataframe e nomi, per comodità
dfs = [df_0, df_1, df_2, df_3, df_4, df_5, df_6]
df_names = [file_name_0, file_name_1, file_name_2, file_name_3, file_name_4, file_name_5, file_name_6]
df_code = ['df_0', 'df_1', 'df_2', 'df_3', 'df_4', 'df_5', 'df_6']

time_buffer = pd.Timedelta(days=90)

for df in dfs:
    df['EXAMDATE'] = pd.to_datetime(df['EXAMDATE'])
    df = df.sort_values(by=['RID', 'EXAMDATE']).reset_index(drop=True)
    


In [7]:

subj_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID'])
print("righe con stessi ####### RID:")
display(subj_matrix)
        
subj_date_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE'], time_buffer=time_buffer)
print("righe con stessi ####### RID-EXAMDATE: --> time_buffer=", time_buffer)
display(subj_date_matrix)

subj_viscode_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'VISCODE'])
print("righe con stessi ####### RID-VISCODE:")
display(subj_viscode_matrix)

righe con stessi ####### RID:


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,844,,,,,,
df_1,11,807,,,,,
df_2,64,281,1122,,,,
df_3,82,73,318,1067,,,
df_4,755,11,63,82,755,,
df_5,840,11,64,82,753,840,
df_6,735,11,64,78,679,734,736


righe con stessi ####### RID-EXAMDATE: --> time_buffer= 90 days 00:00:00


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,4143,,,,,,
df_1,0,847,,,,,
df_2,0,0,2222,,,,
df_3,5,0,0,4350,,,
df_4,3503,0,0,3,3504,,
df_5,839,0,0,0,746,840,
df_6,2569,0,0,2,2474,733,2597


righe con stessi ####### RID-VISCODE:


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,4087,,,,,,
df_1,0,845,,,,,
df_2,0,0,2220,,,,
df_3,5,0,0,4346,,,
df_4,3493,0,0,3,3501,,
df_5,838,0,0,0,744,840,
df_6,2567,0,0,2,2470,734,2597


# Inizio Merge
## Definizione di df_merge_0 e Gerarchia di DF da mergiare
Scegliere il file con numero maggiore di soggetti-visite e che ha più elementi con altri df.\
Quindi scegliere con che ordine unire gli altri df, suggerimento da quelli con nessuna/pochissime righe RID-EXAMDATE in comune con gli altri df, e quindi quelli con molte righe in comune a partire da quello con più righe in comune sia con df_merge_0 che con gli altri e quindi a seguire. Ma di persè il metodo è arbitrario quindi si può fare come si vuole.

In [8]:
df_merge_0 = df_0.copy(deep=True)
to_merge_hierarchy = [df_2, df_1, df_3, df_4, df_6, df_5]
time_buffer = pd.Timedelta(days=30)
i = 0

## Studio relazioni tra due file

In [9]:
df_to_add = to_merge_hierarchy[3].copy(deep=True)
i += 1
print(i)

1


### Numero soggetti comuni e match con EXAMDATE e VISITCODE

### Approfondimento RID-EXAMDATE
1. vedo quante righe ci sono con match ESATTO e quante con TIME BUFFER.

In [10]:
buff_index1, buff_index2 = mergeTools.compare_matches_with_without_buffer(df_merge_0, df_to_add, time_buffer=time_buffer, print_info=True)
all_index1, all_index2 = mergeTools.get_date_match_index(df_merge_0, df_to_add, time_buffer=time_buffer)

[date_matches_with_buffer] Trovati solo match esatti --> 3503.
[date_matches_with_buffer] Trovati solo match esatti --> 3503.
ZERO matches con TIME BUFFER


Sezione in cui studio le righe con match non esatto ma con TIME BUFFER \
==> cerco di capire il perchè. \
Se è perchè un dataset contine visite specifiche tenuta in data diversa della visita generale allora cambio nome EXAMDATE in merge.\
Se ci sia match esatti che con TIME BUFFER capirne bene il perchè.\

In [11]:
if len(buff_index1) == len(buff_index2) and len(buff_index1) > 0:
    if all(all_index1) == len(all_index2) and all_index1 == buff_index1 and all_index2 == buff_index2:
        print(f'all maches have a buffer, tot: {len(all_index1)} matches')
    else:
        print(f'There are {len(buff_index1)} match with buffer\nThere are {len(all_index1)-len(buff_index1)} match exact\nOver {len(all_index1)} total matches')
    columns_in_common = [x for x in df_merge_0.columns if x in df_to_add.columns]
    columns_only_merge = [x for x in df_merge_0.columns if x not in df_to_add.columns]
    columns_only_add = [x for x in df_to_add.columns if x not in df_merge_0.columns]
    if columns_only_add and columns_only_merge:
        print('Le colonne in comune sono:\n', columns_in_common)
        print('\n\nLe colonne solo in df_merge_0 sono: \n', columns_only_merge)
        print('\n\nLe colonne solo in df_to_add sono: \n', columns_only_add)
elif len(buff_index1) == len(buff_index2) and len(buff_index1) == 0:
    print('JUST exact matches')


JUST exact matches


In [12]:
'''
confrontando le colonne in comune e meno, decido se inserire o meno una nuova colonna chiamata tipo PET_DATE e così via.
se non matchano i bbuffer con i tot match allora valuto se guardare quali variabili sono sempre nan quando ci sono match esatti e quando ci sono match con buffer.

Quindi aggiungop nuova colonna, e copio valori di EXAMDATE, lascio examdate e però agisco quyando mergio per soggetto --> modifica da implementare
'''

'\nconfrontando le colonne in comune e meno, decido se inserire o meno una nuova colonna chiamata tipo PET_DATE e così via.\nse non matchano i bbuffer con i tot match allora valuto se guardare quali variabili sono sempre nan quando ci sono match esatti e quando ci sono match con buffer.\n\nQuindi aggiungop nuova colonna, e copio valori di EXAMDATE, lascio examdate e però agisco quyando mergio per soggetto --> modifica da implementare\n'

### Studio Colonne in comune per righe che matchano
Definite le colonne di riferimento, quindi comuni a più o meno tutti i file, valuto se ci sono righe e colonne in comuni ai due df.\
Se ci sono sia colonne in comune che righe in che machano per RID ed EXAMDATE (anche con buffer) allora è necessario uno studio più approfondito.\
Altrimenti, se o non ci sono colonne in comune o non ci sono righe in comune, allora il merge sarà più semplice. \
E possiamo passare al Merge vero e proprio.

In [13]:
reference_col = ['RID', 'VISCODE', 'EXAMDATE', 'VISIT_MONTH', 'COHORT'] #più altre specifiche

common_columns = [x for x in list(df_merge_0.columns) if x in list(df_to_add.columns) and x not in reference_col]
common_columns


['STATUS',
 'ICV%ICV',
 'MidTemp%ICV',
 'Fusiform%ICV',
 'Ventricles%ICV',
 'Entorhinal%ICV',
 'Hippocampus%ICV']

In [14]:
if len(all_index1) != len(all_index2):
    print("C'è un problema nell'identificazione dei match, gli indici di match tra df_merge e df_to_add deevono coincidere")
elif len(all_index1) > 0 and len(common_columns) > 0:
    print('Necessario confrontare i risultati delle colonne comuni per righe comuni, DEFINIRE REGOLE da seguire per merge queste righe')
    print('colonne comuni --> ', common_columns)
elif len(common_columns) == 0:
    print('EASY! Puoi mergiare direttamente, NON si sovrappongono COLONNE a parte quelle di riferimento')
elif len(all_index1) == 0:
    print('EASY! Puoi mergiare direttamente, NON si sovrappongono RIGHE')

Necessario confrontare i risultati delle colonne comuni per righe comuni, DEFINIRE REGOLE da seguire per merge queste righe
colonne comuni -->  ['STATUS', 'ICV%ICV', 'MidTemp%ICV', 'Fusiform%ICV', 'Ventricles%ICV', 'Entorhinal%ICV', 'Hippocampus%ICV']


In [15]:
rid_list = df_merge_0.iloc[all_index1]['RID'].unique().tolist()
print(len(rid_list))

755


In [16]:
df_a = df_merge_0[~df_merge_0['RID'].isin(rid_list)].copy(deep=True)
df_b = df_to_add[~df_to_add['RID'].isin(rid_list)].copy(deep=True)

df_merge_iniziale = pd.merge(df_a, df_b, how='outer')

In [ ]:
ref_col = [x for x in reference_col if x in df_merge_0.columns and x in df_to_add.columns and x != 'RID']
df_merge = df_merge_iniziale.copy(deep=True)
for rid in rid_list:
    merged_sub_df = mergeTools.merge_paired_rows_rid_specific(df_merge_0, df_to_add, all_index1, all_index2, ref_col, subject_id=rid)
    df_merge = pd.merge(df_merge, merged_sub_df, how='outer')
    print(rid, len(merged_sub_df), len(df_merge))


3
3 4 114
4
4 5 119
5
5 5 124
6
6 5 129
7
7 3 132
8
8 2 134
10
10 4 138
14
14 5 143
15
15 4 147
16
16 5 152
19
19 4 156
21
21 5 161
22
22 4 165
23
23 6 171
29
29 4 175
30
30 6 181
31
31 10 191
33
33 6 197
35
35 4 201
38
38 3 204
40
### VISCODE is not a reference column studied, it will be merged with the simple method
40 6 210
41
41 7 217
42
42 11 228
43
43 6 234
44
44 2 236
45
45 2 238
47
47 4 242
48
48 5 247
50
50 5 252
51
51 9 261
53
53 4 265
54
54 5 270
55
55 7 277
56
56 6 283
57
57 5 288
58
58 8 296
59
59 6 302
60
60 4 306
61
61 11 317
66
66 5 322
67
67 4 326
68
68 7 333
69
69 6 339
70
70 3 342
72
72 9 351
74
74 11 362
76
76 4 366
77
77 5 371
78
78 4 375
80
80 3 378
81
81 6 384
83
83 4 388
84
84 4 392
86
86 6 398
87
87 4 402
88
88 3 405
89
89 6 411
90
90 5 416
91
91 4 420
93
93 3 423
94
94 4 427
95
95 3 430
96
96 8 438
97
97 4 442
98
98 4 446
101
101 9 455
102
102 5 460
103
103 2 462
106
106 9 471
107
107 8 479
108
108 9 488
109
109 4 492
110
110 2 494
111
111 3 497
112
112 10 507

In [20]:
df_merge

,RID,VISCODE,VISIT_MONTH,EXAMDATE,STATUS,ICV%ICV,MidTemp%ICV,Fusiform%ICV,Ventricles%ICV,Entorhinal%ICV,Hippocampus%ICV
0,2,sc,0.0,2005-08-26,complete,100.0,1.407596,0.834349,5.702085,0.210464,0.420022
1,3,m06,6.0,2006-03-13,complete,100.0,0.929958,0.772045,3.948375,0.123634,0.273338
2,3,m12,12.0,2006-09-12,complete,100.0,0.911404,0.767247,4.008914,0.113981,0.270876
3,3,m24,24.0,2007-09-12,complete,100.0,0.859217,0.729503,4.334277,0.105704,0.260505
4,3,sc,0.0,2005-09-01,complete,100.0,0.949294,0.826370,3.753130,0.122404,0.273730
...,...,...,...,...,...,...,...,...,...,...,...
4139,1427,sc,0.0,2007-08-20,complete,100.0,1.522409,1.100454,1.226945,0.236754,0.566323
4140,1427,None,84.0,2014-08-18,complete,100.0,1.441502,1.032535,1.709435,0.225522,0.474024
4141,1430,m06,7.0,2008-04-04,complete,100.0,1.110064,1.052849,1.934094,0.209734,0.336206
4142,1430,sc,0.0,2007-09-07,complete,100.0,1.099489,1.073505,1.865333,0.214495,0.337881


In [19]:
print(len(df_a), len(df_b))
print('somma', len(df_a) + len(df_b))
print('merge', len(df_merge_iniziale))
print('merge finale', len(df_merge))

110 0
somma 110
merge 110
merge finale 4144


In [ ]:
import json

# Apri e carica il file diff_tracking.json
with open('diff_tracking.json', 'r') as f:
    diff_tracking = json.load(f)

# Ad esempio, mostriamo le chiavi e il primo valore
print("Chiavi disponibili in diff_tracking.json:", list(diff_tracking.keys()))
for key in diff_tracking.keys():
    print(key, len(key),'/',len(rid_list))


In [ ]:
rid = 40

In [ ]:
df_merge_0[df_merge_0['RID']==30]

In [ ]:
df_to_add[df_to_add['RID']==30]

In [ ]:
ref_col = [x for x in reference_col if x in df_merge_0.columns and x in df_to_add.columns and x != 'RID']
temp_merge = mergeTools.create_temp_merge(df_merge_0, df_to_add, all_index1, all_index2, rid=rid, col_list=ref_col+common_columns)
merged_sub_df = mergeTools.merge_paired_rows_rid_specific(df_merge_0, df_to_add, all_index1, all_index2, ref_col, subject_id=rid)

    

In [ ]:
temp_merge

In [ ]:
[temp_merge['EXAMDATE_1'].dropna(axis=0, how='any')&temp_merge['EXAMDATE_1'].dropna(axis=0, how='any')]

In [ ]:
temp_merge[['ICV%ICV_1','ICV%ICV_2', 'MidTemp%ICV_1', 'MidTemp%ICV_2', 'Fusiform%ICV_1',
       'Fusiform%ICV_2', 'Ventricles%ICV_1', 'Ventricles%ICV_2',
       'Entorhinal%ICV_1', 'Entorhinal%ICV_2', 'Hippocampus%ICV_1',
       'Hippocampus%ICV_2']]

In [ ]:
merged_sub_df

# altro

## Studio soggetti, soggetti-date, soggetti-visite in comune tra i 2 df identificati

In [ ]:
### Ripasso quanti soggetti, e soggetto-visita in comune hanno i 2 df

# Lista dei RID in comune tra i due DataFrame (ad esempio df_merge_0 e il primo da unire)
if 'RID' in df_merge_0.columns and 'RID' in df_to_add.columns:
    subj_in_common = set(df_merge_0['RID']).intersection(df_to_add['RID'])
    print(f"Numero di Soggetti in comune tra df_merge_0 e df da unire: {len(subj_in_common)}")
    print("Esempio di RID in comune:", list(subj_in_common)[:10])
    
    # lista RID-EXAMDATE (+-time_buffer) comuni
    if 'EXAMDATE' in df_merge_0.columns and 'EXAMDATE' in df_to_add.columns:
        subj_date_common_df = mergeTools.date_matches_with_buffer(df_merge_0, df_to_add, time_buffer)
        print(f"\nNumero di Soggetti-examdate in comune tra df_merge_0 e df da unire: {len(subj_date_common_df)}")
        print("Esempio di RID in comune:", subj_date_common_df.head())
    else:
        print("una delle 2 tabelle non ha la colonna 'EXAMDATE'")
    
    # lista RID-VISITCODE comuni
    if 'VISCODE' in df_merge_0.columns and 'VISCODE' in df_to_add.columns:
        subj_viscode_in_common = set(zip(df_merge_0['RID'], df_merge_0['VISCODE'])).intersection(set(zip(df_to_add['RID'], df_to_add['VISCODE'])))
        print(f"\nNumero di Soggetti in comune tra df_merge_0 e df da unire: {len(subj_viscode_in_common)}")
        print("Esempio di RID in comune:", list(subj_viscode_in_common)[:10])
    else:
        print("una delle 2 tabelle non ha la colonna 'VISCODE'")
        subj_viscode_in_common = []
else:
    print("Una delle due tabelle non ha la colonna 'RID'")

In [ ]:
# Trova tutte le coppie (RID, VISCODE) in df_merge_0 i cui (RID, EXAMDATE) corrispondono alle (RID, EXAMDATE_1) presenti in subj_date_common_df
if 'RID' in df_merge_0.columns and 'EXAMDATE' in df_merge_0.columns and 'VISCODE' in df_merge_0.columns:
    # Filtro df_merge_0 sulle tuple (RID, EXAMDATE) presenti in subj_date_common_df
    merge_keys = set(zip(subj_date_common_df['RID'], subj_date_common_df['EXAMDATE_1']))
    mask = df_merge_0.apply(lambda row: (row['RID'], row['EXAMDATE']) in merge_keys, axis=1)
    matched_df_merge = df_merge_0[mask]

    # Estrai le coppie (RID, VISCODE) dalla selezione
    rid_viscode_matched_list = set(zip(matched_df_merge['RID'], matched_df_merge['VISCODE']))
    
    # Confronta questa lista con subj_viscode_in_common
    subj_viscode_in_common_set = set(subj_viscode_in_common)
    not_in_matched = subj_viscode_in_common_set - rid_viscode_matched_list
    
    print(f"Numero di elementi di subj_viscode_in_common NON trovati tra le nuove coppie (RID, VISCODE): {len(not_in_matched)}")
    print("Esempi di coppie (RID, VISCODE) assenti:", list(not_in_matched)[:10])
else:
    print("df_merge_0 non ha tutte le colonne richieste ('RID', 'EXAMDATE', 'VISCODE')")

if len(not_in_matched) > 0:
    # Trova gli indici delle righe in df_merge_0 che corrispondono alle coppie in not_in_matched
    mask_not_in_matched = df_merge_0.apply(lambda row: (row['RID'], row['VISCODE']) in not_in_matched, axis=1)
    df_not_in_matched = df_merge_0[mask_not_in_matched]
    print("Indici delle righe in df_merge_0 corrispondenti alle coppie not_in_matched:")
    print(df_not_in_matched.index.tolist())
    print("Primi esempi delle righe:")
    print(df_not_in_matched.head())
else:
    print("Tutte le coppie (RID, VISCODE) di subj_viscode_in_common sono presenti tra le nuove coppie.")


In [ ]:
mergeTools.individual_match_and_missing_rows(df_0, dfs=[df_1, df_2, df_3, df_4, df_5, df_6], df_names=df_code, columns_list=['RID', 'EXAMDATE'])

In [ ]:
# Lista dei dataframe e nomi, per comodità
dfs = [df_0, df_1, df_2, df_3, df_4, df_5, df_6]
df_names = [file_name_0, file_name_1, file_name_2, file_name_3, file_name_4, file_name_5, file_name_6]
df_code = ['df_0', 'df_1', 'df_2', 'df_3', 'df_4', 'df_5', 'df_6']

# Assicurati che le colonne RID e VISCODE siano presenti e che EXAMVISCODEDATE sia in formato datetime
for i, df in enumerate(dfs):
    if 'VISCODE' not in df.columns:
        print(f"AVVISO: '{df_names[i]}' non contiene colonna VISCODE")
    if 'RID' not in df.columns:
        print(f"AVVISO: '{df_names[i]}' non contiene colonna RID")

# Matrice di match per (RID, VISCODE)
n = len(dfs)
match_matrix = pd.DataFrame(0, index=df_code, columns=df_code)

for i in range(n):
    for j in range(n):
        # Ci assicuriamo di confrontare solo se entrambe le colonne esistono nei dati
        if {'RID', 'VISCODE'}.issubset(dfs[i].columns) and {'RID', 'VISCODE'}.issubset(dfs[j].columns):
            s1 = set(dfs[i][['RID', 'VISCODE']].drop_duplicates().itertuples(index=False, name=None))
            s2 = set(dfs[j][['RID', 'VISCODE']].drop_duplicates().itertuples(index=False, name=None))
            match_matrix.iloc[i, j] = len(s1 & s2)
        else:
            match_matrix.iloc[i, j] = None  # Indica colonne mancanti


display_match_matrix = match_matrix.copy()
mask = np.triu(np.ones(display_match_matrix.shape), k=1).astype(bool)
display_match_matrix = display_match_matrix.mask(mask, "")

print("Matrice del numero di righe con stessi RID (solo diagonale inferiore):")
display(display_match_matrix)





In [ ]:
# Lista dei dataframe e nomi, per comodità
dfs = [df_0, df_1, df_2, df_3, df_4, df_5, df_6]
df_names = [file_name_0, file_name_1, file_name_2, file_name_3, file_name_4, file_name_5, file_name_6]
df_code = ['df_0', 'df_1', 'df_2', 'df_3', 'df_4', 'df_5', 'df_6']

# Definisci il time_buffer (in giorni) per considerare match anche con date leggermente diverse
time_buffer = pd.Timedelta(days=0)  # Modifica questo valore secondo le tue esigenze (es. days=7 per una settimana)

# Assicurati che le colonne RID e EXAMDATE siano presenti e che EXAMDATE sia in formato datetime
for i, df in enumerate(dfs):
    if 'EXAMDATE' in df.columns:
        dfs[i]['EXAMDATE'] = pd.to_datetime(df['EXAMDATE'])
    else:
        print(f"AVVISO: '{df_names[i]}' non contiene colonna EXAMDATE")
    if 'RID' not in df.columns:
        print(f"AVVISO: '{df_names[i]}' non contiene colonna RID")

# Funzione helper per contare i match con buffer temporale


# Matrice di match per (RID, EXAMDATE) con buffer temporale
n = len(dfs)
match_matrix = pd.DataFrame(0, index=df_code, columns=df_code)

for i in range(n):
    for j in range(n):
        # Ci assicuriamo di confrontare solo se entrambe le colonne esistono nei dati
        if {'RID', 'EXAMDATE'}.issubset(dfs[i].columns) and {'RID', 'EXAMDATE'}.issubset(dfs[j].columns):
            match_matrix.iloc[i, j] = len(mergeTools.count_matches_with_buffer(dfs[i], dfs[j], time_buffer))
        else:
            match_matrix.iloc[i, j] = None  # Indica colonne mancanti


display_match_matrix = match_matrix.copy()
mask = np.triu(np.ones(display_match_matrix.shape), k=1).astype(bool)
display_match_matrix = display_match_matrix.mask(mask, "")

print("Matrice del numero di righe con stessi RID (solo diagonale inferiore):")
display(display_match_matrix)




In [ ]:
# Seleziona solo le righe in cui TUTTI i volumi richiesti NON sono NaN
vol_cols = [
    "Ventricles%ICV",
    "Hippocampus%ICV",
    "Entorhinal%ICV",
    "Fusiform%ICV",
    "MidTemp%ICV",
    "ICV%ICV"
]
df_merge_vol = df_merge.dropna(subset=vol_cols, how="any").copy()

# Di che colonne identificative vogliamo il confronto?
# Ragionevolmente, 'RID' (subject id) -- eseguiamo il confronto su questo campo.

# Set di RIDs nei due dataframe
rids_merge_vol = set(df_merge_vol["RID"].unique())
rids_vol = set(df_vol["RID"].unique())

# 1) Quanti soggetti in comune hanno i due df
common_rids = rids_merge_vol & rids_vol
print("Numero di soggetti in comune tra df_merge_vol e df_vol:", len(common_rids))

# 2) Quanti sogg ha df_vol che df_merge_vol non ha
only_in_vol = rids_vol - rids_merge_vol
print("Numero di soggetti che sono in df_vol ma non in df_merge_vol:", len(only_in_vol))


In [ ]:
print('rows ADNIMERGE', len(df_merge))
print('rows df_vol', len(df_vol))
print('rows df_merge_vol', len(df_merge_vol))


In [ ]:
import pandas as pd

# Assicurati che le colonne EXAMDATE siano in formato datetime
df_vol['EXAMDATE'] = pd.to_datetime(df_vol['EXAMDATE'], errors='coerce')
df_merge_vol['EXAMDATE'] = pd.to_datetime(df_merge_vol['EXAMDATE'], errors='coerce')

# Per velocità, possiamo ordinare i dataframe
df_merge_vol_sorted = df_merge_vol.sort_values(['RID', 'EXAMDATE'])
df_vol_sorted = df_vol.sort_values(['RID', 'EXAMDATE'])

common_count = 0
only_in_vol_count = 0

idxs_in_common = []
idxs_only_in_vol = []

# Per ciascuna riga di df_vol, verifichiamo se esiste una riga dello stesso RID in df_merge_vol con EXAMDATE a +/- 15gg
for idx, row in df_vol_sorted.iterrows():
    rid = row['RID']
    examdate = row['EXAMDATE']
    sub_merge = df_merge_vol_sorted[df_merge_vol_sorted['RID'] == rid]
    # Trova se c'è almeno un examdate entro +/-15 giorni
    mask_in_range = sub_merge[
        (sub_merge['EXAMDATE'] - examdate).abs().dt.days <= 80
    ]
    if not mask_in_range.empty:
        common_count += 1
        idxs_in_common.append(idx)
    else:
        only_in_vol_count += 1
        idxs_only_in_vol.append(idx)

print(f"Numero di righe in comune tra df_vol e df_merge_vol (stesso RID e EXAMDATE a +/-15gg): {common_count}")
print(f"Numero di righe di df_vol che NON hanno match in df_merge_vol con stesso RID ed EXAMDATE a +/-15gg: {only_in_vol_count}")




In [ ]:
# Seleziona le righe di df_vol che NON hanno un match in df_merge_vol con stesso RID ed EXAMDATE a +/-15gg
df_only_in_vol = df_vol_sorted.loc[idxs_only_in_vol]
print("\nPrime 10 righe di df_vol che df_merge_vol non ha (confronto su RID + EXAMDATE +/-80gg):")
display(df_only_in_vol.head(10))

In [ ]:
import numpy as np
# Calcola quanti soggetti (RID) in df_only_in_vol sono anche presenti in df_merge_vol (indipendentemente dalla data)
unique_rid_only_in_vol = df_only_in_vol['RID'].unique()
unique_rid_merge_vol = df_merge_vol['RID'].unique()
unique_rid_merge = df_merge['RID'].unique()

count_rid_in_both = sum(np.isin(unique_rid_only_in_vol, unique_rid_merge_vol))
count_rid_in_original = sum(np.isin(unique_rid_only_in_vol, unique_rid_merge))
print(f"Numero di soggetti (RID) in df_only_in_vol che sono presenti anche in df_merge_vol: {count_rid_in_both} / {len(unique_rid_only_in_vol)}")
print(f"Numero di soggetti (RID) in df_only_in_vol che sono presenti anche in ADNIMERGE: {count_rid_in_original} / {len(unique_rid_only_in_vol)}")
# Trova i soggetti (RID) in df_only_in_vol che non sono mai presenti in ADNIMERGE (unique_rid_merge)
rids_not_in_adnimerge = [rid for rid in unique_rid_only_in_vol if rid not in unique_rid_merge]
print(f"Soggetti in df_only_in_vol CHE NON sono in ADNIMERGE (unique_rid_merge): {rids_not_in_adnimerge}")
print(f"Totale: {len(rids_not_in_adnimerge)}")


In [ ]:
df_out = df_vol[df_vol['RID'].isin(rids_not_in_adnimerge)]
len(df_out)


In [ ]:
df_merge_vol[df_merge_vol['RID']==123]